# Compresión intermedia: expertos RCAB en los niveles 1 y 2 de la U-Net

Punto intermedio entre `S2_d32`, que sustituye sólo las RCAB del nivel 2, y
`S2_d32_l012`, que sustituye las de los tres niveles. Lleva el mismo módulo de
ancho 32, `1×1 → GELU → depthwise 3×3 → 1×1`, a las 16 RCAB de encoder y
decoder de los niveles 1 y 2, en las dos etapas. El nivel 0, el de resolución
completa, conserva sus convoluciones originales:

- `S2_d32_l12`: un banco de cinco módulos por RCAB, uno por tarea; se ejecuta sólo el de la tarea.
- `S1_d32_l12`: un módulo común por RCAB. Es el control de mismo costo.

**Por qué.** `S2_d32_l012` quedó 0,378 dB por debajo del profesor en PSNR macro
de validación, con dehaze −1,82 y derain −0,42. Esta corrida dice si la pérdida
viene del nivel 0. Costo a 256×256, programa sin optimizar: `S2_d32_l12` tiene
11,85 M parámetros (−16,4 % frente a B0) y 182,6 GFLOP (−15,6 %), contra
−13,5 % y −7,3 % de `S2_d32` y −16,8 % y −23,3 % de `S2_d32_l012`.

Lo único que cambia frente a `S1_d32`/`S2_d32` son los niveles sustituidos. El
protocolo de adaptación es idéntico, incluido su hash, y estas variantes se
registran en este notebook, sin tocar `maxim.py` ni `compression.py`.

**Orden sugerido.** Correr primero `S2_d32_l12` con `REPLICA = "v1"` y después
`S1_d32_l12` como control. La segunda semilla queda para después, si la
variante se confirma.

**Criterio de decisión, fijado antes de correr.** El mismo de la opción 1, sólo
con validación: el test ya no es ciego para variantes nuevas. Es favorable si el
mejor `S2_d32_l12` queda a menos de 0,05 dB del profesor en PSNR macro de
validación y ninguna tarea cae más de 0,10 dB frente a él. La última celda
calcula ambos chequeos y compara con las corridas v1 de `S1_d32`, `S2_d32`,
`S1_d32_l012` y `S2_d32_l012`, que deben estar en `ckpts/compresion_adaptacion/`.

**Protocolo.** Es el de `adaptacion_compresion.ipynb`:

- Profesor: shared +10 con selección efectiva, congelado.
- Alumno: igual al profesor salvo por los módulos nuevos, cuya salida arranca
  en cero. Arranca como el modelo sin esas 16 parejas de convoluciones.
- Pérdida: L1 con supervisión profunda, destilación L1 de la salida final y
  destilación de las salidas de las 16 RCAB sustituidas.
- Fases: 5 épocas de 2.000 pasos entrenando sólo los módulos nuevos.
- Selección: mejor PSNR macro de validación, incluida la época 0. **Nunca se usa test.**

**GPU.** Una L4, como las corridas de referencia. Cada corrida lleva unas 1,4 h.
Reanudación igual que antes: reconectar y ejecutar todo.


In [ ]:
# 1. Entorno: fork con los módulos de compresión
import hashlib
import os
import subprocess
import sys
from pathlib import Path

# MAXIM_NOTEBOOK_SMOKE=1 sólo se usa para validar el notebook localmente en CPU
# con un modelo reducido y datos sintéticos. En Colab queda en False.
SMOKE_TEST = os.environ.get("MAXIM_NOTEBOOK_SMOKE") == "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Margen para el paso de entrenamiento, que pide ~12 GiB en un solo bloque.
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")

REPO_URL = "https://github.com/Matiata/maxim.git"
REPO_REF = "main"  # rama o commit con maxim/models/compression.py
EXPECTED_SOURCE_SHA256 = {
    "maxim/models/maxim.py": "5f8a4e005ffc5b12b5dcfbd9b2a06cab0125dbbdb97813c4b1f8ea1b2d5bf57a",
    "maxim/models/compression.py": "b460019514b05dd0b0bbe828f2d145aa12dbc443e36554b0fc735dbc84636c4c",
    "maxim/models/compat.py": "cebfaa8bcd5eaf947c3d98962a3e2266e44265b16f0836b2d5c878581abe610c"
}


def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args], text=True
    ).strip()


if SMOKE_TEST:
    REPO_DIR = Path(os.environ["MAXIM_REPO_DIR"]).resolve()
else:
    REPO_DIR = Path("/content/maxim_compresion_src")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif git("remote", "get-url", "origin").rstrip("/") != REPO_URL:
        raise RuntimeError("REPO_DIR apunta a otro repositorio.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    try:
        target = git("rev-parse", "--verify", f"origin/{REPO_REF}")
    except subprocess.CalledProcessError:
        target = REPO_REF
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--detach", target], check=True
    )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "jax[cuda12]==0.10.2", "flax==0.11.2", "optax==0.2.8",
            "ml-collections==0.1.1", "einops", "Pillow",
        ],
        check=True,
    )

try:
    REPO_COMMIT = git("rev-parse", "HEAD")
except Exception as error:
    REPO_COMMIT = f"unavailable: {error}"

SOURCE_SHA256 = {}
for relative, expected in EXPECTED_SOURCE_SHA256.items():
    path = REPO_DIR / relative
    if not path.exists():
        raise FileNotFoundError(
            f"Falta {relative} en {REPO_REF}. Subir los cambios (git push)."
        )
    SOURCE_SHA256[relative] = hashlib.sha256(path.read_bytes()).hexdigest()
    if SOURCE_SHA256[relative] != expected:
        raise RuntimeError(
            f"{relative} no coincide con la versión revisada: "
            f"{SOURCE_SHA256[relative]} != {expected}."
        )

sys.path.insert(0, str(REPO_DIR))
print("Commit:", REPO_COMMIT, "| modo smoke:", SMOKE_TEST)
print("Si JAX ya estaba importado antes de instalar, reiniciar el runtime.")

In [ ]:
# 2. Drive, dependencias y GPU
if not SMOKE_TEST:
    from google.colab import drive

    drive.mount("/content/gdrive")

import collections
import gc
import importlib.metadata
import json
import platform
import re
import shutil
import time
import traceback
from collections.abc import Mapping

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import traverse_util
from flax.core import unfreeze
from flax.training import checkpoints
from PIL import Image

from maxim.evaluation_metrics import (
    aggregate_task_psnr,
    aggregate_task_ssim,
    build_best_metric_payload,
)
from maxim.models.compression import (
    VARIANTS,
    build_variant,
    flatten_params,
    is_new_local_mixer,
    removed_rcab_conv_at,
    transplant_params,
    variant_replaced_levels,
)
from maxim.models.maxim import RCAB

expected_versions = {"jax": "0.10.2", "flax": "0.11.2"}
actual_versions = {
    name: importlib.metadata.version(name) for name in expected_versions
}
if actual_versions != expected_versions:
    raise RuntimeError(f"Versiones inesperadas: {actual_versions}; reiniciar.")
GPU_DEVICES = [device for device in jax.devices() if device.platform == "gpu"]
if not GPU_DEVICES and not SMOKE_TEST:
    raise RuntimeError("Seleccionar un runtime GPU en Colab.")
DEVICE = GPU_DEVICES[0] if GPU_DEVICES else jax.devices()[0]
DEVICE_KIND = getattr(DEVICE, "device_kind", DEVICE.platform)
print("Dispositivo:", DEVICE, DEVICE_KIND)

import tensorflow as tf

tf.config.set_visible_devices([], "GPU")  # TF sólo lee datos; la GPU es de JAX

In [ ]:
# 3. Configuración: elegir la variante; el resto es el protocolo común
VARIANT = "S2_d32_l12"  # @param ["S2_d32_l12", "S1_d32_l12"]
# Réplica: v1 es la corrida original. v2 repite el mismo protocolo con
# otras semillas de datos, dropout e inicialización de los módulos nuevos.
REPLICA = "v1"  # @param ["v1", "v2"]
REPLICA_SEEDS = {
    "v1": {"seed": 42, "new_module_seed": 1234},
    "v2": {"seed": 7, "new_module_seed": 4321},
}
RUN_TAG = REPLICA

# Compresión intermedia: el mismo módulo de S1_d32/S2_d32, ahora en los
# niveles 1 y 2. Las variantes l012 se registran sólo para calcular su costo y
# leer sus corridas en la comparación. Se registran acá para no cambiar
# compression.py, cuyo hash fijan los notebooks ya corridos.
LEVEL_VARIANTS = {
    f"{prefix}_d32_{tag}": {
        "family": "shared_selective", "init": "shared_plus_new",
        "maxim": {"rcab_mixer": "bottleneck", "rcab_mixer_levels": levels,
                  "rcab_mixer_width": 32, "rcab_mixer_num_experts": experts}}
    for tag, levels in (("l12", (1, 2)), ("l012", (0, 1, 2)))
    for prefix, experts in (("S1", 1), ("S2", 5))
}
VARIANTS.update(LEVEL_VARIANTS)
TRAINABLE_VARIANTS = ("S2_d32_l12", "S1_d32_l12")
REFERENCE_RUNS = {name: f"{name}_v1" for name in
                  ("S1_d32", "S2_d32", "S1_d32_l012", "S2_d32_l012")}

TASKS = ["deblur", "dehaze", "denoise", "derain", "enhance"]
if SMOKE_TEST:
    DATA_ROOT = Path(os.environ["MAXIM_SMOKE_DATA"])
    CHECKPOINT_ROOT = Path(os.environ["MAXIM_SMOKE_ROOT"])
    LOCAL_CACHE = Path(os.environ["MAXIM_SMOKE_ROOT"]) / "cache"
else:
    DATA_ROOT = Path("/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier")
    CHECKPOINT_ROOT = Path("/content/gdrive/MyDrive/Facultad/tesis/ckpts")
    LOCAL_CACHE = Path("/content/cache_compresion")
OUTPUT_DIR = CHECKPOINT_ROOT / "compresion_adaptacion" / f"{VARIANT}_{RUN_TAG}"

MODEL_CONFIG = {
    "variant": "S-2", "dropout_rate": 0.1, "num_outputs": 3,
    "use_bias": True, "num_supervision_scales": 3,
}
SHARED_SPEC = {
    "relative_run": "moe_shared_residual_from_newsplit_baseline_extra10",
    "epoch": 8, "expected_step": 16000, "val_macro": 32.06335479306456,
}

# Protocolo común a S1 y S2. Cambiarlo invalida la comparación.
PHASES = [
    {"name": "modulos_nuevos", "scope": "new_only", "epochs": 5,
     "learning_rate": 1e-3, "warmup_steps": 1000, "weight_decay": 1e-4},
    # Fase opcional, igual para S1 y S2, sólo si validación muestra que hace falta:
    # {"name": "todo_el_modelo", "scope": "full", "epochs": 3,
    #  "learning_rate": 1e-5, "warmup_steps": 1000, "weight_decay": 1e-4},
]
STEPS_PER_EPOCH = 2000
BATCH_SIZE = 2
PATCH_SIZE = 256
EVAL_PATCH_SIZE = 256
SEED = REPLICA_SEEDS[REPLICA]["seed"]
NEW_MODULE_SEED = REPLICA_SEEDS[REPLICA]["new_module_seed"]
ZERO_INIT_NEW_OUTPUT = True
AUXILIARY_LOSS_WEIGHT = 1.0
DISTILL_OUTPUT_WEIGHT = 1.0
FEATURE_DISTILL_WEIGHT = 0.1
CLIP_NORM = 1.0
STUDENT_REMAT_BLOCKS = False  # True baja la memoria de la fase completa
LOG_EVERY = 100
TEACHER_VAL_TOLERANCE_DB = 0.01

if SMOKE_TEST:
    MODEL_CONFIG.update(features=8, block_size_hr=(4, 4), grid_size_hr=(4, 4),
                        block_size_lr=(2, 2), grid_size_lr=(2, 2))
    PHASES = [
        {"name": "modulos_nuevos", "scope": "new_only", "epochs": 2,
         "learning_rate": 1e-3, "warmup_steps": 2, "weight_decay": 1e-4},
        {"name": "todo_el_modelo", "scope": "full", "epochs": 1,
         "learning_rate": 1e-5, "warmup_steps": 1, "weight_decay": 1e-4},
    ]
    STEPS_PER_EPOCH, PATCH_SIZE, EVAL_PATCH_SIZE, LOG_EVERY = 3, 32, 32, 1
    SMOKE_STOP_AFTER_EPOCH = int(os.environ.get("MAXIM_SMOKE_STOP_AFTER", "0"))

if VARIANT not in TRAINABLE_VARIANTS:
    raise ValueError(f"Variante no prevista: {VARIANT}")
REPLACED_LEVELS = variant_replaced_levels(VARIANT)
STUDENT_CONFIG = dict(MODEL_CONFIG, remat_blocks=STUDENT_REMAT_BLOCKS)

TRAINING_PROTOCOL = {
    "schema_version": 1,
    "source": SHARED_SPEC,
    "teacher": "S0_sel (shared +10, selección efectiva), congelado, train=False",
    "model_config": MODEL_CONFIG,
    "phases": PHASES,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "batch_size": BATCH_SIZE,
    "patch_size": PATCH_SIZE,
    "eval_patch_size": EVAL_PATCH_SIZE,
    "seed": SEED,
    "new_module_seed": NEW_MODULE_SEED,
    "zero_init_new_output": ZERO_INIT_NEW_OUTPUT,
    "loss": {
        "reconstruction": "L1 final + L1 de supervisión profunda normalizada",
        "auxiliary_weight": AUXILIARY_LOSS_WEIGHT,
        "distill_output_weight": DISTILL_OUTPUT_WEIGHT,
        "feature_distill_weight": FEATURE_DISTILL_WEIGHT,
        "feature_distill": "MSE / energía del profesor, salidas de las RCAB sustituidas",
    },
    "optimizer": "clip_by_global_norm + AdamW; momentos nuevos en cada fase",
    "clip_norm": CLIP_NORM,
    "sampling": "uniforme por tarea; orden y aumentaciones deterministas por época",
    "selection": "psnr_macro de validación, incluida la época 0",
    "student_dropout": MODEL_CONFIG["dropout_rate"],
    "student_remat_blocks": STUDENT_REMAT_BLOCKS,
}
# Las fases forman parte del protocolo, pero se permite agregar fases al final.
PROTOCOL_WITHOUT_PHASES = {k: v for k, v in TRAINING_PROTOCOL.items()
                           if k != "phases"}
TOTAL_EPOCHS = sum(phase["epochs"] for phase in PHASES)
print(f"Réplica {REPLICA} (semillas {SEED}, {NEW_MODULE_SEED}) | "
      f"Variante {VARIANT} | niveles sustituidos {REPLACED_LEVELS} | "
      f"{TOTAL_EPOCHS} épocas | salida {OUTPUT_DIR}")

In [ ]:
# 4. Utilidades: JSON atómico, hashes, entorno y listas de datos
def json_safe(value):
    if isinstance(value, Mapping):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    if isinstance(value, (float, int, str, bool)) or value is None:
        return value
    return str(value)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2,
                   allow_nan=False),
        encoding="utf-8",
    )
    os.replace(temporary, path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def stable_hash(payload):
    return hashlib.sha256(
        json.dumps(json_safe(payload), sort_keys=True).encode("utf-8")
    ).hexdigest()


def hash_params(tree):
    digest = hashlib.sha256()
    for key, value in sorted(flatten_params(tree).items()):
        array = np.ascontiguousarray(np.asarray(jax.device_get(value)))
        if not np.isfinite(array).all():
            raise FloatingPointError(f"Parámetro no finito: {key}")
        digest.update(json.dumps([str(part) for part in key]).encode("utf-8"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(str(array.shape).encode("ascii"))
        digest.update(array.tobytes())
    return digest.hexdigest()


def environment_payload():
    packages = {}
    for name in ("jax", "jaxlib", "flax", "optax", "orbax-checkpoint",
                 "numpy", "tensorflow"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "packages": packages,
        "device": str(DEVICE),
        "device_kind": DEVICE_KIND,
        "repo_commit": REPO_COMMIT,
        "source_sha256": SOURCE_SHA256,
        "mem_fraction": os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"),
        "smoke_test": SMOKE_TEST,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


def read_split_names(task, split):
    path = DATA_ROOT / task / f"{split}.txt"
    raw = path.read_bytes()
    names = [line.strip() for line in raw.decode("utf-8").splitlines()
             if line.strip()]
    if len(names) != len(set(names)):
        raise ValueError(f"Entradas duplicadas en {path}")
    for name in names:
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Ruta inválida en {path}: {name}")
    return names, {"count": len(names),
                   "sha256": hashlib.sha256(raw).hexdigest()}


def verify_split_manifest(reference_run_metadata):
    """Las listas actuales deben ser idénticas a las de la run shared fuente."""
    reference = reference_run_metadata["split_manifest"]
    manifest = {}
    for task in TASKS:
        manifest[task] = {}
        for split in ("train", "val", "test"):
            _, value = read_split_names(task, split)
            wanted = {key: reference[task][split][key]
                      for key in ("count", "sha256")}
            if value != wanted and not SMOKE_TEST:
                raise ValueError(
                    f"Lista distinta de la run fuente: {task}/{split}: "
                    f"{value} != {wanted}")
            manifest[task][split] = value
    return manifest


PROTOCOL_HASH = stable_hash(TRAINING_PROTOCOL)
COMPARABLE_PROTOCOL_HASH = stable_hash(PROTOCOL_WITHOUT_PHASES)
print("Protocolo:", PROTOCOL_HASH)

In [ ]:
# Métricas idénticas a las de los notebooks de entrenamiento y de test
SSIM_DATA_RANGE, SSIM_FILTER_SIZE, SSIM_FILTER_SIGMA = 1.0, 11, 1.5
SSIM_K1, SSIM_K2 = 0.01, 0.03


def compute_psnr_per_example(pred, target):
    diff = pred * 255.0 - target * 255.0
    mse = jnp.mean(diff ** 2, axis=(1, 2, 3))
    mse = jnp.maximum(mse, 1e-6)
    return 20.0 * jnp.log10(255.0 / jnp.sqrt(mse))


def compute_ssim_per_example(pred, target):
    """Single-scale RGB SSIM matching tf.image.ssim defaults."""
    pred = jnp.asarray(pred, dtype=jnp.float32)
    target = jnp.asarray(target, dtype=jnp.float32)
    if pred.ndim != 4 or pred.shape != target.shape:
        raise ValueError(f"SSIM expects equal NHWC tensors, got {pred.shape} "
                         f"and {target.shape}.")
    coordinates = jnp.arange(SSIM_FILTER_SIZE, dtype=jnp.float32)
    coordinates -= (SSIM_FILTER_SIZE - 1.0) / 2.0
    gaussian_1d = jnp.exp(-0.5 * (coordinates / SSIM_FILTER_SIGMA) ** 2)
    gaussian_1d /= jnp.sum(gaussian_1d)
    gaussian_2d = gaussian_1d[:, None] * gaussian_1d[None, :]
    channels = pred.shape[-1]
    kernel = jnp.tile(gaussian_2d[:, :, None, None], (1, 1, 1, channels))

    def depthwise_filter(images):
        return jax.lax.conv_general_dilated(
            images, kernel, window_strides=(1, 1), padding="VALID",
            dimension_numbers=("NHWC", "HWIO", "NHWC"),
            feature_group_count=channels,
        )

    mean_pred = depthwise_filter(pred)
    mean_target = depthwise_filter(target)
    variance_pred = depthwise_filter(pred * pred) - mean_pred * mean_pred
    variance_target = (depthwise_filter(target * target)
                       - mean_target * mean_target)
    covariance = depthwise_filter(pred * target) - mean_pred * mean_target
    c1 = (SSIM_K1 * SSIM_DATA_RANGE) ** 2
    c2 = (SSIM_K2 * SSIM_DATA_RANGE) ** 2
    luminance = (2.0 * mean_pred * mean_target + c1) / (
        mean_pred ** 2 + mean_target ** 2 + c1)
    contrast_structure = (2.0 * covariance + c2) / (
        variance_pred + variance_target + c2)
    return jnp.mean(luminance * contrast_structure, axis=(1, 2, 3))


_probe = np.linspace(0, 1, 2 * 16 * 16 * 3, dtype=np.float32).reshape(2, 16, 16, 3)
np.testing.assert_allclose(
    np.asarray(jax.jit(compute_ssim_per_example)(jnp.asarray(_probe),
                                                 jnp.asarray(_probe))),
    1.0, atol=1e-5)
print("Métricas sintéticas OK")

In [ ]:
# 5. Datos de entrenamiento: deterministas por semilla y época
def random_crop(image, target, crop_size, rng):
    """Mismo recorte alineado que los notebooks previos, con RNG explícito."""
    pads = []
    for array in (image, target):
        h, w = array.shape[:2]
        pads.append((max(0, crop_size - h), max(0, crop_size - w)))
    if any(pads[0]):
        image = np.pad(image, ((0, pads[0][0]), (0, pads[0][1]), (0, 0)),
                       mode="reflect")
    if any(pads[1]):
        target = np.pad(target, ((0, pads[1][0]), (0, pads[1][1]), (0, 0)),
                        mode="reflect")
    h = min(image.shape[0], target.shape[0])
    w = min(image.shape[1], target.shape[1])
    top = int(rng.integers(0, h - crop_size + 1))
    left = int(rng.integers(0, w - crop_size + 1))
    return (image[top:top + crop_size, left:left + crop_size],
            target[top:top + crop_size, left:left + crop_size])


def augment(image, target, rng):
    if rng.random() > 0.5:
        image, target = np.fliplr(image), np.fliplr(target)
    if rng.random() > 0.5:
        image, target = np.flipud(image), np.flipud(target)
    k = int(rng.integers(0, 4))
    return np.rot90(image, k=k), np.rot90(target, k=k)


def load_float(path):
    return load_uint8(path).astype(np.float32) / 255.0


def load_train_pair(input_path, target_path, index, task_id, seed):
    rng = np.random.default_rng(
        [int(seed), int(task_id.numpy()), int(index.numpy())])
    x = load_float(input_path.numpy().decode("utf-8"))
    y = load_float(target_path.numpy().decode("utf-8"))
    x, y = random_crop(x, y, PATCH_SIZE, rng)
    x, y = augment(x, y, rng)
    if x.shape != (PATCH_SIZE, PATCH_SIZE, 3) or y.shape != x.shape:
        raise ValueError(f"Crop inválido: {input_path.numpy()}")
    return np.ascontiguousarray(x), np.ascontiguousarray(y)


def train_file_lists():
    lists = {}
    for task in TASKS:
        names, _ = read_split_names(task, "train")
        inputs = [str(DATA_ROOT / task / "imgs" / name) for name in names]
        targets = [str(DATA_ROOT / task / "GT" / name) for name in names]
        missing = [p for p in inputs + targets if not os.path.exists(p)]
        if missing:
            raise FileNotFoundError(f"{task}: faltan {len(missing)} archivos, "
                                    f"por ejemplo {missing[:3]}")
        lists[task] = (inputs, targets)
        print(f"Train {task}: {len(names)} pares")
    return lists


def train_batches(epoch):
    """2.000 batches de la época `epoch`, idénticos para S1 y S2."""
    seed = SEED * 1000 + int(epoch)
    sources = []
    for task_id, task in enumerate(TASKS):
        inputs, targets = TRAIN_FILES[task]
        ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
        ds = ds.shuffle(len(inputs), seed=seed + 17 * task_id,
                        reshuffle_each_iteration=True).repeat()
        ds = ds.enumerate()
        ds = ds.map(
            lambda index, pair, task_id=task_id: tuple(tf.py_function(
                load_train_pair,
                [pair[0], pair[1], index, task_id, seed],
                [tf.float32, tf.float32])) + (tf.constant(task_id, tf.int32),),
            num_parallel_calls=min(4, os.cpu_count() or 1), deterministic=True)
        sources.append(ds)
    mixed = tf.data.Dataset.sample_from_datasets(
        sources, weights=[1.0 / len(TASKS)] * len(TASKS), seed=seed,
        stop_on_empty_dataset=False)
    mixed = mixed.map(
        lambda x, y, t: (tf.ensure_shape(x, [PATCH_SIZE, PATCH_SIZE, 3]),
                         tf.ensure_shape(y, [PATCH_SIZE, PATCH_SIZE, 3]), t))
    mixed = mixed.batch(BATCH_SIZE, drop_remainder=True).take(STEPS_PER_EPOCH)
    return mixed.prefetch(tf.data.AUTOTUNE).as_numpy_iterator()

In [ ]:
# Validación en memoria: crop central 256×256, igual que en entrenamiento y test
from concurrent.futures import ThreadPoolExecutor

def center_crop(image, target, crop_size):
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]
    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect")
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect")
    h = min(image.shape[0], target.shape[0])
    w = min(image.shape[1], target.shape[1])
    top = max(0, (h - crop_size) // 2)
    left = max(0, (w - crop_size) // 2)
    # Copias: un recorte de NumPy es una vista que mantiene viva la imagen
    # completa. Sin copiar, el caché retenía todas las imágenes originales.
    return (image[top:top + crop_size, left:left + crop_size].copy(),
            target[top:top + crop_size, left:left + crop_size].copy())


def load_uint8(path, retries=5):
    last_error = None
    for attempt in range(retries):
        try:
            with Image.open(path) as image:
                return np.asarray(image.convert("RGB"), dtype=np.uint8)
        except OSError as error:
            last_error = error
            time.sleep(0.3)
    raise OSError(f"No se pudo leer {path}") from last_error


LOAD_WORKERS = 8  # lecturas en paralelo desde Drive


def _load_crop_pair(task, name):
    x = load_uint8(DATA_ROOT / task / "imgs" / name)
    y = load_uint8(DATA_ROOT / task / "GT" / name)
    x, y = center_crop(x, y, EVAL_PATCH_SIZE)
    if x.shape != (EVAL_PATCH_SIZE, EVAL_PATCH_SIZE, 3) or y.shape != x.shape:
        raise ValueError(f"Shape inválida en {task}/{name}: {x.shape}")
    return x, y


def build_task_cache(split, task):
    """Crops centrales de una tarea, en uint8 y en el orden de la lista."""
    task_names, _ = read_split_names(task, split)
    started = time.time()
    inputs, targets = [], []
    with ThreadPoolExecutor(max_workers=LOAD_WORKERS) as pool:
        pairs = pool.map(lambda name: _load_crop_pair(task, name), task_names)
        for count, (x, y) in enumerate(pairs, 1):
            inputs.append(x)
            targets.append(y)
            if count % 250 == 0 or count == len(task_names):
                print(f"  {split}/{task}: {count}/{len(task_names)} pares | "
                      f"{time.time() - started:.0f} s", flush=True)
    return np.stack(inputs), np.stack(targets), task_names


def load_or_build_val_cache(cache_path, manifest_hash, split="val"):
    """Un archivo por tarea: si el runtime se corta, sólo se rehace la tarea
    incompleta. Dividir por 255 da los mismos float32 que el pipeline original,
    que convertía a float antes de recortar."""
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    xs, ys, task_ids, names = [], [], [], []
    for task_id, task in enumerate(TASKS):
        task_path = cache_path.with_name(f"{cache_path.stem}_{task}.npz")
        data = None
        if task_path.exists():
            data = np.load(task_path, allow_pickle=False)
            if str(data["manifest_hash"]) != manifest_hash:
                data = None
        if data is None:
            x, y, task_names = build_task_cache(split, task)
            np.savez(task_path, manifest_hash=manifest_hash, x=x, y=y,
                     names=np.asarray(task_names))
            del x, y
            data = np.load(task_path, allow_pickle=False)
        else:
            print(f"{split}/{task}: reutilizado {task_path}")
        xs.append(data["x"])
        ys.append(data["y"])
        task_ids.append(np.full(len(data["names"]), task_id, dtype=np.int32))
        names += [f"{task}/{name}" for name in data["names"]]
    return {"x": np.concatenate(xs), "y": np.concatenate(ys),
            "task_id": np.concatenate(task_ids), "names": names}


def summarize_per_image(psnr, ssim, task_ids):
    psnr, ssim = np.asarray(psnr, np.float64), np.asarray(ssim, np.float64)
    task_ids = np.asarray(task_ids)
    counts = [int(np.sum(task_ids == t)) for t in range(len(TASKS))]
    psnr_sum = [float(np.sum(psnr[task_ids == t])) for t in range(len(TASKS))]
    ssim_sum = [float(np.sum(ssim[task_ids == t])) for t in range(len(TASKS))]
    p = aggregate_task_psnr(psnr_sum, counts)
    s = aggregate_task_ssim(ssim_sum, counts)
    return {
        "psnr_macro": float(p["psnr_macro"]),
        "psnr_weighted": float(p["psnr_weighted"]),
        "ssim_macro": float(s["ssim_macro"]),
        "ssim_weighted": float(s["ssim_weighted"]),
        "task_psnr": [float(v) for v in p["task_psnr"]],
        "task_ssim": [float(v) for v in s["task_ssim"]],
        "task_count": counts,
        "sample_count": int(sum(counts)),
    }

In [ ]:
# Restauración estricta de checkpoints de referencia
def variant_family(name):
    return VARIANTS[name.split("@")[0]]["family"]


def template_params(model, name):
    kwargs = ({} if variant_family(name) == "baseline"
              else {"task_id": jnp.zeros((1,), dtype=jnp.int32)})
    dummy = jnp.zeros((1, 64, 64, 3), dtype=jnp.float32)
    if SMOKE_TEST:
        dummy = jnp.zeros((1, 32, 32, 3), dtype=jnp.float32)
    return jax.eval_shape(lambda: model.init(
        jax.random.PRNGKey(0), dummy, train=False, **kwargs))["params"]


def validate_tree(expected, actual, label):
    wanted, found = flatten_params(expected), flatten_params(actual)
    missing = sorted(set(wanted) - set(found))
    extra = sorted(set(found) - set(wanted))
    if missing or extra:
        raise ValueError(f"{label}: faltan {missing[:4]}, sobran {extra[:4]}")
    for key in wanted:
        if (tuple(wanted[key].shape) != tuple(found[key].shape)
                or np.dtype(wanted[key].dtype) != np.dtype(found[key].dtype)):
            raise ValueError(f"{label}: shape o dtype incompatible en {key}")


def restore_reference_params(spec, template):
    run_dir = CHECKPOINT_ROOT / spec["relative_run"]
    selection = read_json(run_dir / "best_checkpoint" / "best_metric.json")
    if selection.get("selection_metric") != "psnr_macro":
        raise ValueError(f"{spec['relative_run']}: selección no macro.")
    if int(selection["epoch"]) != spec["epoch"]:
        raise ValueError(f"{spec['relative_run']}: cambió la época elegida.")
    if abs(float(selection["psnr_macro"]) - spec["val_macro"]) > 1e-6:
        raise ValueError(f"{spec['relative_run']}: cambió el PSNR elegido.")
    raw = checkpoints.restore_checkpoint(
        ckpt_dir=str(run_dir / "best_checkpoint" / f"checkpoint_{spec['epoch']}"),
        target=None)
    if not isinstance(raw, Mapping) or "params" not in raw or "step" not in raw:
        raise ValueError("No se restauró un TrainState completo.")
    if int(np.asarray(raw["step"])) != spec["expected_step"]:
        raise ValueError(f"Paso inesperado en {spec['relative_run']}.")
    validate_tree(template, raw["params"], spec["relative_run"])
    params = jax.tree_util.tree_map(jnp.asarray, raw["params"])
    return params, {
        "checkpoint": str(run_dir / "best_checkpoint" / f"checkpoint_{spec['epoch']}"),
        "restored_step": int(np.asarray(raw["step"])),
        "params_sha256": hash_params(params),
        "selection": selection,
        "run_metadata": read_json(run_dir / "run_metadata.json"),
    }



# 6. Profesor (shared con selección efectiva) y alumno (variante comprimida)
LEVEL_PATTERN = re.compile(r"stage_\d+_(?:encoder|decoder)_block_(\d+)$")


def is_replaced_rcab(module, method_name):
    """Filtro de capture_intermediates: salidas de las RCAB sustituidas."""
    if method_name != "__call__" or not isinstance(module, RCAB):
        return False
    for part in module.scope.path:
        match = LEVEL_PATTERN.match(str(part))
        if match and int(match.group(1)) in REPLACED_LEVELS:
            return True
    return False


SHARED_RUN_METADATA = read_json(
    CHECKPOINT_ROOT / SHARED_SPEC["relative_run"] / "run_metadata.json")
SPLIT_MANIFEST = verify_split_manifest(SHARED_RUN_METADATA)

TEACHER = build_variant("S0_sel", MODEL_CONFIG, num_tasks=len(TASKS))
TEACHER_PARAMS, TEACHER_SOURCE = restore_reference_params(
    SHARED_SPEC, template_params(TEACHER, "S0_sel"))
STUDENT = build_variant(VARIANT, STUDENT_CONFIG, num_tasks=len(TASKS))


def initial_student_params():
    dummy = jnp.zeros((1, 64, 64, 3) if not SMOKE_TEST else (1, 32, 32, 3))
    fresh = jax.jit(lambda key: STUDENT.init(
        key, dummy, train=False, task_id=jnp.zeros((1,), jnp.int32))
    )(jax.random.PRNGKey(NEW_MODULE_SEED))["params"]
    params, report = transplant_params(
        TEACHER_PARAMS, fresh,
        allowed_new=is_new_local_mixer,
        allowed_removed=removed_rcab_conv_at(REPLACED_LEVELS))
    flat = flatten_params(params)
    if ZERO_INIT_NEW_OUTPUT:
        for key in list(flat):
            if "local_mixer" in key and key[-1] in ("out_kernel", "out_bias"):
                flat[key] = jnp.zeros_like(flat[key])
    params = traverse_util.unflatten_dict(flat)
    validate_tree(template_params(STUDENT, VARIANT), params, VARIANT)
    teacher_flat = flatten_params(TEACHER_PARAMS)
    copied = [key for key in flat if "local_mixer" not in key]
    if not all(np.array_equal(np.asarray(flat[key]), np.asarray(teacher_flat[key]))
               for key in copied):
        raise AssertionError("El alumno no copió exactamente los pesos del profesor.")
    report = {k: v for k, v in report.items() if not k.endswith("_paths")}
    # Copia propia: el alumno no debe compartir buffers con el profesor, porque
    # el paso de entrenamiento dona los parámetros entrenables.
    return jax.tree_util.tree_map(lambda a: jnp.array(a, copy=True), params), report


def count(tree):
    return int(sum(np.prod(v.shape) for v in jax.tree_util.tree_leaves(tree)))


print("Profesor:", count(TEACHER_PARAMS), "parámetros |",
      TEACHER_SOURCE["params_sha256"][:12])

In [ ]:
# 7. Pérdidas, paso de entrenamiento y evaluación
def resize_target_to(pred, target):
    if pred.shape == target.shape:
        return target
    return target[:, ::target.shape[1] // pred.shape[1],
                  ::target.shape[2] // pred.shape[2], :]


def compute_supervision_losses(predictions, targets, num_scales=3,
                               auxiliary_weight=1.0):
    final_prediction = predictions[-1][-1]
    final_loss = jnp.mean(jnp.abs(
        final_prediction - resize_target_to(final_prediction, targets)))
    auxiliary_sum = jnp.zeros_like(final_loss)
    weight_sum = 0.0
    last_stage = len(predictions) - 1
    for stage_index, stage_predictions in enumerate(predictions):
        for scale_index, prediction in enumerate(stage_predictions):
            if (stage_index == last_stage
                    and scale_index == len(stage_predictions) - 1):
                continue
            weight = 0.5 ** (num_scales - scale_index - 1)
            auxiliary_sum += weight * jnp.mean(jnp.abs(
                prediction - resize_target_to(prediction, targets)))
            weight_sum += weight
    auxiliary_loss = auxiliary_sum / max(weight_sum, 1e-8)
    return (final_loss + auxiliary_weight * auxiliary_loss, final_loss,
            auxiliary_loss, final_prediction)


def captured_features(state):
    flat = traverse_util.flatten_dict(unfreeze(state["intermediates"]))
    return {key: value[0] if isinstance(value, tuple) else value
            for key, value in flat.items()}


def feature_distillation(student_features, teacher_features):
    if set(student_features) != set(teacher_features) or not teacher_features:
        raise ValueError("Las salidas capturadas de alumno y profesor no coinciden.")
    losses = []
    for key, target in teacher_features.items():
        target = jax.lax.stop_gradient(target)
        error = jnp.mean((student_features[key] - target) ** 2)
        losses.append(error / (jnp.mean(target ** 2) + 1e-6))
    return jnp.mean(jnp.stack(losses))


def split_params(params, scope):
    flat = flatten_params(params)
    if scope == "full":
        return params, {}
    if scope != "new_only":
        raise ValueError(scope)
    trainable = {k: v for k, v in flat.items() if "local_mixer" in k}
    frozen = {k: v for k, v in flat.items() if "local_mixer" not in k}
    return (traverse_util.unflatten_dict(trainable),
            traverse_util.unflatten_dict(frozen))


def merge_params(trainable, frozen):
    merged = dict(flatten_params(frozen)) if frozen else {}
    merged.update(flatten_params(trainable))
    return traverse_util.unflatten_dict(merged)


def make_optimizer(phase):
    steps = phase["epochs"] * STEPS_PER_EPOCH
    warmup = min(phase["warmup_steps"], max(steps - 1, 0))
    schedule = optax.join_schedules(
        [optax.linear_schedule(0.0, phase["learning_rate"], max(warmup, 1)),
         optax.cosine_decay_schedule(phase["learning_rate"],
                                     max(steps - warmup, 1), alpha=1e-3)],
        boundaries=[warmup])
    return optax.chain(
        optax.clip_by_global_norm(CLIP_NORM),
        optax.adamw(learning_rate=schedule,
                    weight_decay=phase["weight_decay"]))


def make_train_step(tx):
    num_scales = MODEL_CONFIG["num_supervision_scales"]

    def step(trainable, frozen, opt_state, teacher_params, x, y, task_id, rng):
        (teacher_preds, _), teacher_state = TEACHER.apply(
            {"params": teacher_params}, x, train=False, task_id=task_id,
            capture_intermediates=is_replaced_rcab, mutable=["intermediates"])
        teacher_final = jax.lax.stop_gradient(teacher_preds[-1][-1])
        teacher_features = captured_features(teacher_state)

        def loss_fn(trainable):
            params = merge_params(trainable, frozen)
            (preds, _), state = STUDENT.apply(
                {"params": params}, x, train=True, task_id=task_id,
                rngs={"dropout": rng}, capture_intermediates=is_replaced_rcab,
                mutable=["intermediates"])
            reconstruction, final_loss, auxiliary, final = (
                compute_supervision_losses(preds, y, num_scales,
                                           AUXILIARY_LOSS_WEIGHT))
            distill = jnp.mean(jnp.abs(final - teacher_final))
            features = feature_distillation(captured_features(state),
                                            teacher_features)
            total = (reconstruction + DISTILL_OUTPUT_WEIGHT * distill
                     + FEATURE_DISTILL_WEIGHT * features)
            return total, {"loss": total, "reconstruction": reconstruction,
                           "final_l1": final_loss, "auxiliary_l1": auxiliary,
                           "distill_output": distill,
                           "distill_features": features}

        (_, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(trainable)
        updates, opt_state = tx.update(grads, opt_state, trainable)
        trainable = optax.apply_updates(trainable, updates)
        metrics["grad_norm"] = optax.global_norm(grads)
        return trainable, opt_state, metrics

    return jax.jit(step, donate_argnums=(0, 2))


@jax.jit
def student_eval_step(params, x, y, task_id):
    preds, _ = STUDENT.apply({"params": params}, x, train=False, task_id=task_id)
    final = preds[-1][-1]
    return compute_psnr_per_example(final, y), compute_ssim_per_example(final, y)


@jax.jit
def teacher_eval_step(params, x, y, task_id):
    preds, _ = TEACHER.apply({"params": params}, x, train=False, task_id=task_id)
    final = preds[-1][-1]
    return compute_psnr_per_example(final, y), compute_ssim_per_example(final, y)


def evaluate(step_fn, params, label):
    started = time.time()
    psnr, ssim = [], []
    for index in range(len(VAL["task_id"])):
        x = jnp.asarray(VAL["x"][index:index + 1].astype(np.float32) / 255.0)
        y = jnp.asarray(VAL["y"][index:index + 1].astype(np.float32) / 255.0)
        t = jnp.asarray(VAL["task_id"][index:index + 1])
        p, s = jax.device_get(step_fn(params, x, y, t))
        psnr.append(float(p[0]))
        ssim.append(float(s[0]))
    if not np.isfinite(psnr).all() or not np.isfinite(ssim).all():
        raise FloatingPointError(f"Métrica no finita en {label}.")
    summary = summarize_per_image(psnr, ssim, VAL["task_id"])
    print(f"[val {label}] macro {summary['psnr_macro']:.4f} dB | ponderado "
          f"{summary['psnr_weighted']:.4f} dB | SSIM macro "
          f"{summary['ssim_macro']:.4f} | {time.time() - started:.0f} s")
    return summary

In [ ]:
# 8. Crear o reanudar la corrida
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
BEST_DIR = OUTPUT_DIR / "best_checkpoint"
RUN_METADATA_PATH = OUTPUT_DIR / "run_metadata.json"
HISTORY_PATH = OUTPUT_DIR / "history.json"
LOG_PATH = OUTPUT_DIR / "training.log"
PHASE_BOUNDS = np.cumsum([0] + [phase["epochs"] for phase in PHASES])


def phase_of_epoch(epoch):
    """Fase de la época `epoch` (1-based)."""
    for index in range(len(PHASES)):
        if PHASE_BOUNDS[index] < epoch <= PHASE_BOUNDS[index + 1]:
            return index
    raise ValueError(epoch)


TRAIN_FILES = train_file_lists()
VAL = load_or_build_val_cache(
    LOCAL_CACHE / f"val_{EVAL_PATCH_SIZE}.npz",
    stable_hash({t: SPLIT_MANIFEST[t]["val"] for t in TASKS}))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_METADATA_PATH.exists():
    RUN_METADATA = read_json(RUN_METADATA_PATH)
    if RUN_METADATA["variant"] != VARIANT:
        raise ValueError("OUTPUT_DIR pertenece a otra variante.")
    if RUN_METADATA["variant_spec"] != json_safe(VARIANTS[VARIANT]):
        raise ValueError("Cambió la definición de la variante. Usar otro RUN_TAG.")
    if RUN_METADATA["comparable_protocol_hash"] != COMPARABLE_PROTOCOL_HASH:
        raise ValueError("Cambió el protocolo de la corrida. Usar otro RUN_TAG.")
    stored_phases = RUN_METADATA["protocol"]["phases"]
    if json_safe(PHASES[:len(stored_phases)]) != stored_phases:
        raise ValueError("Sólo se pueden agregar fases al final, no modificarlas.")
    RUN_METADATA["protocol"] = json_safe(TRAINING_PROTOCOL)
    RUN_METADATA["protocol_hash"] = PROTOCOL_HASH
    HISTORY = read_json(HISTORY_PATH)
    latest = checkpoints.latest_checkpoint(str(CHECKPOINT_DIR))
    if latest is None:
        raise FileNotFoundError("Hay metadata pero no checkpoints; revisar OUTPUT_DIR.")
    raw = checkpoints.restore_checkpoint(ckpt_dir=latest, target=None)
    START_EPOCH = int(np.asarray(raw["epoch"]))
    params = jax.tree_util.tree_map(jnp.asarray, raw["params"])
    validate_tree(template_params(STUDENT, VARIANT), params, "checkpoint")
    RESUME_OPT_STATE = None
    if START_EPOCH < TOTAL_EPOCHS and START_EPOCH > 0:
        phase_index = phase_of_epoch(START_EPOCH + 1)
        if phase_of_epoch(START_EPOCH) == phase_index:
            trainable, _ = split_params(params, PHASES[phase_index]["scope"])
            template = {"opt_state": make_optimizer(PHASES[phase_index]).init(trainable)}
            RESUME_OPT_STATE = jax.tree_util.tree_map(
                jnp.asarray,
                checkpoints.restore_checkpoint(
                    ckpt_dir=latest, target=dict(raw, **template))["opt_state"])
    del raw
    print(f"Reanudando {VARIANT} desde la época {START_EPOCH}/{TOTAL_EPOCHS}")
else:
    params, transplant_report = initial_student_params()
    START_EPOCH, RESUME_OPT_STATE = 0, None
    RUN_METADATA = {
        "variant": VARIANT,
        "variant_spec": json_safe(VARIANTS[VARIANT]),
        "run_tag": RUN_TAG,
        "protocol": json_safe(TRAINING_PROTOCOL),
        "protocol_hash": PROTOCOL_HASH,
        "comparable_protocol_hash": COMPARABLE_PROTOCOL_HASH,
        "split_manifest": SPLIT_MANIFEST,
        "teacher_source": {k: v for k, v in TEACHER_SOURCE.items()
                           if k != "run_metadata"},
        "transplant": transplant_report,
        "parameters": {"total": count(params),
                       "new_modules": count(split_params(params, "new_only")[0])},
        "sessions": [],
    }
    HISTORY = []
    print("Evaluando el profesor para verificar el protocolo de validación...")
    teacher_val = evaluate(teacher_eval_step, TEACHER_PARAMS, "profesor")
    difference = abs(teacher_val["psnr_macro"] - SHARED_SPEC["val_macro"])
    RUN_METADATA["teacher_validation"] = teacher_val
    RUN_METADATA["teacher_validation_diff_db"] = difference
    if difference > TEACHER_VAL_TOLERANCE_DB and not SMOKE_TEST:
        raise AssertionError(
            f"La validación del profesor no reproduce la registrada: "
            f"{teacher_val['psnr_macro']:.4f} vs {SHARED_SPEC['val_macro']:.4f}")
    initial_val = evaluate(student_eval_step, params, "alumno, época 0")
    HISTORY.append({"epoch": 0, "phase": None, "validation": initial_val})
    BEST_DIR.mkdir(parents=True, exist_ok=True)
    checkpoints.save_checkpoint(
        ckpt_dir=str(BEST_DIR), target={"params": params, "step": 0, "epoch": 0},
        step=0, keep=1, overwrite=True)
    atomic_json(BEST_DIR / "best_metric.json", build_best_metric_payload(
        0, initial_val, TASKS, "psnr_macro"))
    checkpoints.save_checkpoint(
        ckpt_dir=str(CHECKPOINT_DIR),
        target={"params": params, "step": 0, "epoch": 0}, step=0, keep=2,
        overwrite=True)
    atomic_json(HISTORY_PATH, HISTORY)

RUN_METADATA["sessions"].append(environment_payload())
atomic_json(RUN_METADATA_PATH, RUN_METADATA)
BEST_VALUE = read_json(BEST_DIR / "best_metric.json")["psnr_macro"]
print(f"Mejor macro hasta ahora: {BEST_VALUE:.4f} dB")

In [ ]:
# 9. Entrenamiento por épocas, con validación y checkpoint en cada una
class TeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
        return len(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()


_console = sys.stdout
_log_file = open(LOG_PATH, "a", encoding="utf-8", buffering=1)
sys.stdout = TeeStream(_console, _log_file)
try:
    print("\n" + "=" * 70)
    print(f"Sesión {time.strftime('%Y-%m-%d %H:%M:%S')} | {VARIANT} | "
          f"GPU {DEVICE_KIND} | desde época {START_EPOCH}")
    current_phase, tx, train_step = None, None, None
    trainable = frozen = opt_state = None
    frozen_hash = None
    for epoch in range(START_EPOCH + 1, TOTAL_EPOCHS + 1):
        phase_index = phase_of_epoch(epoch)
        phase = PHASES[phase_index]
        if phase_index != current_phase:
            trainable, frozen = split_params(params, phase["scope"])
            # Buffers propios para lo que se dona en cada paso.
            trainable = jax.tree_util.tree_map(
                lambda a: jnp.array(a, copy=True), trainable)
            tx = make_optimizer(phase)
            if RESUME_OPT_STATE is not None:
                opt_state, RESUME_OPT_STATE = RESUME_OPT_STATE, None
            else:
                opt_state = tx.init(trainable)
            train_step = make_train_step(tx)
            frozen_hash = hash_params(frozen) if frozen else None
            current_phase = phase_index
            print(f"Fase {phase['name']} ({phase['scope']}): entrenables "
                  f"{count(trainable):,} | congelados "
                  f"{count(frozen) if frozen else 0:,}")
        started = time.time()
        sums = collections.defaultdict(float)
        for step, (x, y, t) in enumerate(train_batches(epoch)):
            rng = jax.random.fold_in(jax.random.PRNGKey(SEED), epoch * 100003 + step)
            trainable, opt_state, metrics = train_step(
                trainable, frozen, opt_state, TEACHER_PARAMS,
                jnp.asarray(x), jnp.asarray(y), jnp.asarray(t), rng)
            if (step + 1) % LOG_EVERY == 0 or step == 0:
                metrics = jax.device_get(metrics)
                for key, value in metrics.items():
                    sums[key] += float(value)
                sums["_n"] += 1
                print(f"época {epoch} paso {step + 1}: " + ", ".join(
                    f"{k}={float(v):.4f}" for k, v in metrics.items()))
        if step + 1 != STEPS_PER_EPOCH:
            raise RuntimeError(f"La época {epoch} tuvo {step + 1} pasos.")
        params = merge_params(trainable, frozen)
        validation = evaluate(student_eval_step, params, f"época {epoch}")
        record = {
            "epoch": epoch, "phase": phase["name"], "scope": phase["scope"],
            "train_logged_mean": {k: v / sums["_n"] for k, v in sums.items()
                                  if k != "_n"},
            "validation": validation,
            "seconds": time.time() - started,
            "device_kind": DEVICE_KIND,
        }
        HISTORY.append(record)
        state_to_save = {"params": params, "opt_state": opt_state,
                         "step": epoch * STEPS_PER_EPOCH, "epoch": epoch}
        checkpoints.save_checkpoint(
            ckpt_dir=str(CHECKPOINT_DIR), target=state_to_save, step=epoch,
            keep=2, overwrite=True)
        if validation["psnr_macro"] > BEST_VALUE:
            BEST_VALUE = validation["psnr_macro"]
            checkpoints.save_checkpoint(
                ckpt_dir=str(BEST_DIR),
                target={"params": params, "step": epoch * STEPS_PER_EPOCH,
                        "epoch": epoch},
                step=epoch, keep=1, overwrite=True)
            atomic_json(BEST_DIR / "best_metric.json", build_best_metric_payload(
                epoch, validation, TASKS, "psnr_macro"))
            print(f"Nuevo mejor: {BEST_VALUE:.4f} dB en la época {epoch}")
        last_of_phase = epoch == PHASE_BOUNDS[phase_index + 1]
        if last_of_phase and frozen_hash is not None:
            if hash_params(frozen) != frozen_hash:
                raise AssertionError("Los parámetros congelados cambiaron.")
            record["frozen_parameters_unchanged"] = True
            print("Verificado: los parámetros congelados no cambiaron.")
        atomic_json(HISTORY_PATH, HISTORY)
        if SMOKE_TEST and SMOKE_STOP_AFTER_EPOCH and epoch == SMOKE_STOP_AFTER_EPOCH:
            print("Smoke: corte simulado de la sesión.")
            break
    RUN_METADATA["completed_epochs"] = HISTORY[-1]["epoch"]
    RUN_METADATA["best"] = read_json(BEST_DIR / "best_metric.json")
    atomic_json(RUN_METADATA_PATH, RUN_METADATA)
    print(f"Fin de sesión | épocas completas {HISTORY[-1]['epoch']}/{TOTAL_EPOCHS}"
          f" | mejor macro {BEST_VALUE:.4f} dB")
finally:
    sys.stdout = _console
    _log_file.close()

In [ ]:
# 10. Resumen por época
rows = ["| Época | Fase | PSNR macro | PSNR ponderado | SSIM macro | "
        + " | ".join(TASKS) + " |",
        "|---:|---|---:|---:|---:|" + "---:|" * len(TASKS)]
for record in HISTORY:
    v = record["validation"]
    rows.append(f"| {record['epoch']} | {record.get('phase') or 'inicial'} | "
                f"{v['psnr_macro']:.4f} | {v['psnr_weighted']:.4f} | "
                f"{v['ssim_macro']:.4f} | "
                + " | ".join(f"{p:.3f}" for p in v["task_psnr"]) + " |")
teacher = RUN_METADATA.get("teacher_validation")
if teacher:
    rows.append(f"| profesor | — | {teacher['psnr_macro']:.4f} | "
                f"{teacher['psnr_weighted']:.4f} | {teacher['ssim_macro']:.4f} | "
                + " | ".join(f"{p:.3f}" for p in teacher["task_psnr"]) + " |")
summary = "\n".join(rows)
(OUTPUT_DIR / "resumen.md").write_text(summary + "\n", encoding="utf-8")
print(summary)

In [ ]:
# 11. Comparación con las corridas de referencia y costo
# Sólo validación. Las referencias son las corridas v1 de S1_d32, S2_d32,
# S1_d32_l012 y S2_d32_l012, con el mismo protocolo y las mismas semillas.
CRITERION_MACRO_DB = 0.05
CRITERION_TASK_DB = 0.10


def best_of(history):
    return max(history, key=lambda record: record["validation"]["psnr_macro"])


def forward_gflop(name):
    """GFLOP del programa sin optimizar, a 256×256, batch 1, como el benchmark.

    Se baja y se analiza en CPU. Con la GPU como dispositivo por defecto, JAX
    analiza con el backend de la GPU aunque se baje para CPU, y ese backend no
    tiene análisis previo a la optimización; el del ejecutable no cuenta las
    convoluciones de cuDNN. Sólo usa shapes.
    """
    model = build_variant(name, MODEL_CONFIG, num_tasks=len(TASKS))
    shapes = template_params(model, name)
    size = EVAL_PATCH_SIZE
    x = jax.ShapeDtypeStruct((1, size, size, 3), jnp.float32)
    with jax.default_device(jax.devices("cpu")[0]):
        if variant_family(name) == "baseline":
            apply = lambda p, x: model.apply({"params": p}, x, train=False)
            traced = jax.jit(apply).trace(shapes, x)
        else:
            t = jax.ShapeDtypeStruct((1,), jnp.int32)
            apply = lambda p, x, t: model.apply(
                {"params": p}, x, train=False, task_id=t)
            traced = jax.jit(apply).trace(shapes, x, t)
        analysis = traced.lower().cost_analysis()
    if isinstance(analysis, (list, tuple)):
        analysis = analysis[0]
    if not analysis or "flops" not in analysis:
        print(f"Sin análisis de FLOPs para {name}.")
        return None, count(shapes)
    return float(analysis["flops"]) / 1e9, count(shapes)


teacher_val = RUN_METADATA["teacher_validation"]
rows = []
for name in ("B0", "S0_sel", "S1_d32", "S2_d32", *LEVEL_VARIANTS):
    gflop, parameters = forward_gflop(name)
    row = {"model": name, "parameters": parameters, "gflop_256": gflop}
    if name == "S0_sel":
        row.update(epoch=None, validation=teacher_val)
    run_dir = (OUTPUT_DIR.parent / f"{name}_{RUN_TAG}" if name in TRAINABLE_VARIANTS
               else OUTPUT_DIR.parent / REFERENCE_RUNS.get(name, "-"))
    if (run_dir / "history.json").exists():
        best = best_of(read_json(run_dir / "history.json"))
        row.update(epoch=best["epoch"], validation=best["validation"])
    rows.append(row)

reference_params = next(r["parameters"] for r in rows if r["model"] == "B0")
reference_gflop = next(r["gflop_256"] for r in rows if r["model"] == "B0")
gflop_text = lambda value: (
    "— | —" if value is None or reference_gflop is None else
    f"{value:.1f} | {100 * (value / reference_gflop - 1):+.1f} %")
lines = ["| Modelo | Parámetros | vs B0 | GFLOP 256 | vs B0 | Época | PSNR macro "
         "| Δ vs profesor | Ponderado | " + " | ".join(TASKS) + " |",
         "|---|---:|---:|---:|---:|---:|---:|---:|---:|" + "---:|" * len(TASKS)]
for row in rows:
    v = row.get("validation")
    quality = ("| — | — | — | — | " + " | ".join("—" for _ in TASKS) if v is None else
               f"| {row['epoch'] if row['epoch'] is not None else '—'} | "
               f"{v['psnr_macro']:.4f} | "
               f"{v['psnr_macro'] - teacher_val['psnr_macro']:+.3f} | "
               f"{v['psnr_weighted']:.4f} | "
               + " | ".join(f"{p:.3f}" for p in v["task_psnr"]))
    lines.append(
        f"| {row['model']} | {row['parameters'] / 1e6:.3f} M | "
        f"{100 * (row['parameters'] / reference_params - 1):+.1f} % | "
        f"{gflop_text(row['gflop_256'])} "
        + quality + " |")

current = next(r for r in rows if r["model"] == VARIANT)
best_val = current["validation"]
macro_delta = best_val["psnr_macro"] - teacher_val["psnr_macro"]
task_deltas = [s - t for s, t in zip(best_val["task_psnr"], teacher_val["task_psnr"])]
decision = {
    "variant": VARIANT,
    "best_epoch": current["epoch"],
    "macro_delta_vs_teacher_db": macro_delta,
    "task_delta_vs_teacher_db": dict(zip(TASKS, task_deltas)),
    "macro_within_criterion": macro_delta >= -CRITERION_MACRO_DB,
    "tasks_within_criterion": min(task_deltas) >= -CRITERION_TASK_DB,
}
decision["favorable"] = (decision["macro_within_criterion"]
                         and decision["tasks_within_criterion"])
report = "\n".join([
    f"# Compresión intermedia: {VARIANT}, réplica {REPLICA}", "",
    "Validación, crop central 256. GFLOP: programa sin optimizar, batch 1. "
    "S0_sel es el profesor evaluado en esta corrida; B0 sólo aporta el costo.", "",
    *lines, "",
    f"Criterio: Δ macro vs profesor ≥ −{CRITERION_MACRO_DB} dB y ninguna tarea "
    f"por debajo de −{CRITERION_TASK_DB} dB.",
    f"Δ macro {macro_delta:+.3f} dB; peor tarea {min(task_deltas):+.3f} dB "
    f"({TASKS[int(np.argmin(task_deltas))]}). "
    f"**{'Favorable' if decision['favorable'] else 'No favorable'}.**",
])
(OUTPUT_DIR / "comparacion.md").write_text(report + "\n", encoding="utf-8")
atomic_json(OUTPUT_DIR / "comparacion.json", {"rows": rows, "decision": decision})
print(report)


## Entrega

Compartir desde `ckpts/compresion_adaptacion/` la carpeta `S2_d32_l12_v1`
(y `S1_d32_l12_v1` si se corrió) sin los checkpoints: `run_metadata.json`,
`history.json`, `resumen.md`, `comparacion.md`, `comparacion.json` y
`training.log`.

No evaluar en test con estos modelos.
